# 📘 Deploy Application Tools> **Applicable Environment**: Kubernetes Pod (Ubuntu base image)> **Purpose**: Install commonly used application tools needed for development and debugging, including opencode (AI programming assistant) and pbcopy (clipboard transfer tool).## 1. Competition IntroductionThis is an AMD Radeon-hackathon-2026-07 competition project.- **Runtime Constraints**: No Docker/Podman inside the Pod, data persistence via PVC mounts, PostgreSQL managed by custom scripts- **Key Paths**: The persistent directory `/workspace/persistent` is mapped to `/data` for unified service access## 2. Install opencodeDownload and install the opencode AI programming assistant to `/data/app`, and write its path to `~/.profile` for global use.```python%%bash#!/bin/bashset -euo pipefailAPP_DIR="/data/app"OPENCODE_VERSION="v1.18.9"TARBALL="opencode-linux-x64.tar.gz"DOWNLOAD_URL="https://github.com/anomalyco/opencode/releases/download/${OPENCODE_VERSION}/${TARBALL}"# 1. Create app directorymkdir -p "$APP_DIR"echo "📁 App Directory: $APP_DIR"# 2. Check if already installedif [ -x "$APP_DIR/opencode" ]; then    echo "✅ opencode installed: $($APP_DIR/opencode --version 2>/dev/null || echo 'Ready')"else    # 3. Download opencode tarball    echo "🔧 Starting download of opencode ${OPENCODE_VERSION} ..."    wget "$DOWNLOAD_URL" --no-check-certificate -O "/tmp/${TARBALL}"    echo "✅ Download complete, starting extraction..."    tar -xzf "/tmp/${TARBALL}" -C "$APP_DIR"    rm -f "/tmp/${TARBALL}"    echo "✅ Installation complete"fi# 4. Configure PATH (~/.profile)PROFILE_LINE='export PATH="$PATH:/data/app"'if ! grep -qF "/data/app" "$HOME/.profile" 2>/dev/null; then    echo "$PROFILE_LINE" >> "$HOME/.profile"    echo "✅ /data/app has been written to ~/.profile"else    echo "✅ /data/app is already in ~/.profile"fi# 5. Verify installationif [ -x "$APP_DIR/opencode" ]; then    echo ""    echo "📊 Verification results:"    "$APP_DIR/opencode" --version || echo "opencode is ready"    ls -la "$APP_DIR"else    echo "❌ opencode installation failed, please check network or download URL"    exit 1fi```## 3. Configure pbcopyInstall the `pbcopy` clipboard transfer tool (macOS-compatible style), used to copy content to the local clipboard via terminal OSC 52 sequence.```python%%bash#!/bin/bashset -euo pipefail# 1. Write pbcopy scriptsudo tee /usr/local/bin/pbcopy > /dev/null << 'EOF'#!/bin/shprintf '\033]52;c;%s\a' "$(base64 | tr -d '\n')"EOF# 2. Grant execution permissionsudo chmod +x /usr/local/bin/pbcopy# 3. Verify installationif [ -x /usr/local/bin/pbcopy ]; then    echo "✅ pbcopy installed successfully: $(which pbcopy)"    echo "--- Script content ---"    cat /usr/local/bin/pbcopyelse    echo "❌ pbcopy installation failed"    exit 1fi```## 4. Usage Examples- **opencode**: After executing `source ~/.profile`, directly run `opencode` to start the AI programming assistant.- **pbcopy**: `echo "hello" | pbcopy`, copy content to local clipboard (requires terminal to support OSC 52).## 5. Follow-up Recommendations- **Configure opencode**: Refer to opencode documentation to set up model API Key and configuration files (`~/.config/opencode/`).- **Configure PostgreSQL environment variables**: Refer to `/data/service/pg-unires/README.md` to set `PGUSER`, `PGPASSWORD`, etc.- **Download model files**: Execute `/scripts/download_models.py` to pull Qwen quantized models to `/data/models`.---> ✅ At this point, application tool deployment is complete, you can continue deploying other Uni-Resource Agent components.